# Task 3 — Voice Audio Processing
## Data Preprocessing & Machine Learning — Formative Assessment 2
### GROUP 9

**Subsystem:** Voice/Audio Collection, Augmentation & Feature Extraction

---
### What this notebook does
| Step | Description |
|------|-------------|
| 1 | Import audio-processing libraries (librosa, soundfile) |
| 2 | Scan `data/audio/` for collected `.wav` recordings |
| 3 | Preview waveform and mel-spectrogram of a sample clip |
| 4 | Apply **augmentations**: pitch shift, time stretch, add Gaussian noise |
| 5 | Extract **MFCC + spectral** features from each clip |
| 6 | Save features to `data/processed/audio_features.csv` |

> **Setup instructions for teammates:**  
> 1. Record 3–5 voice clips as `.wav` files (any phrase, ~3 seconds each)  
> 2. Save them to `data/audio/<your_name>/`  
> 3. Run all cells in order  
> 4. Confirm `audio_features.csv` is saved  

> **Recording tip:** Use the Windows Voice Recorder app or `Audacity` (free).  

> **Running on Google Colab?** Use `!pip install librosa soundfile`, then upload your `.wav` files.

## Step 1 — Import Libraries

`librosa` is the standard Python library for audio analysis — it handles loading, augmentation, and feature extraction.  
`soundfile` provides low-level `.wav` read/write.  
A graceful fallback is included — missing librosa skips all audio operations but the notebook still runs.

In [1]:
# Uncomment on Google Colab:
# !pip install librosa soundfile pandas numpy matplotlib -q

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

try:
    import librosa
    import librosa.display
    import soundfile as sf
    LIBROSA_AVAILABLE = True
    print(f"librosa version: {librosa.__version__}")
except ImportError:
    LIBROSA_AVAILABLE = False
    print("librosa not installed.")
    print("Install with:  pip install librosa soundfile")
    print("Notebook will continue but audio operations will be skipped.")

# Paths
ROOT      = Path('/content') if Path('/content').exists() else Path.cwd().resolve().parents[0]
AUDIO_DIR = ROOT / 'data' / 'audio'
OUT_DIR   = ROOT / 'data' / 'processed'
PLOTS_DIR = ROOT / 'outputs' / 'plots'
OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Audio directory: {AUDIO_DIR}  (exists: {AUDIO_DIR.exists()})")

librosa not installed.
Install with:  pip install librosa soundfile
Notebook will continue but audio operations will be skipped.
Audio directory: C:\Users\user\Desktop\kayonga-elvis task-formative2\data\audio  (exists: True)


*▸ **Output above:** librosa version is printed if available. The audio directory path is confirmed.  
If `exists: False`, create `data/audio/<your_name>/` and add `.wav` recordings before continuing.*

## Step 2 — Discover Audio Files

Scan the audio folder recursively for all `.wav` files.  
Print how many clips were found and their paths.

In [2]:
audio_paths = list(AUDIO_DIR.rglob('*.wav'))
print(f"Audio files found: {len(audio_paths)}")

if not audio_paths:
    print("\nNo audio files found. Please add .wav recordings to data/audio/<your_name>/")
    print("Expected structure:")
    print("  data/")
    print("  └── audio/")
    print("      └── <your_name>/")
    print("          ├── clip_01.wav")
    print("          ├── clip_02.wav")
    print("          └── clip_03.wav")
else:
    for p in audio_paths[:8]:
        print(f"  {p}")

Audio files found: 0

No audio files found. Please add .wav recordings to data/audio/<your_name>/
Expected structure:
  data/
  └── audio/
      └── <your_name>/
          ├── clip_01.wav
          ├── clip_02.wav
          └── clip_03.wav


*▸ **Output above:** Lists up to 8 discovered `.wav` file paths.  
Check that your own subfolder appears. If you see 0 files, re-read the setup instructions in the header.*

## Step 3 — Waveform & Spectrogram Preview

Visualising the **waveform** (amplitude over time) and **mel-spectrogram** (frequency energy over time) of a sample clip helps verify that audio was recorded correctly — it should show a clearly non-flat signal.

In [3]:
if not LIBROSA_AVAILABLE or not audio_paths:
    print("Skipping preview — librosa not available or no audio files found.")
else:
    sample_path = audio_paths[0]
    y, sr = librosa.load(str(sample_path), sr=None)
    print(f"Sample: {sample_path.name}")
    print(f"Duration: {len(y)/sr:.2f}s  |  Sample rate: {sr} Hz  |  Samples: {len(y)}")

    fig, axes = plt.subplots(2, 1, figsize=(12, 7))

    # Waveform
    librosa.display.waveshow(y, sr=sr, ax=axes[0], color='#2d6a4f')
    axes[0].set_title(f'Waveform — {sample_path.name}', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel('Amplitude')

    # Mel spectrogram
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    S_dB = librosa.power_to_db(S, ref=np.max)
    img = librosa.display.specshow(S_dB, sr=sr, x_axis='time', y_axis='mel', ax=axes[1], cmap='YlGn')
    axes[1].set_title('Mel Spectrogram (dB)', fontsize=12, fontweight='bold')
    fig.colorbar(img, ax=axes[1], format="%+2.0f dB")

    fig.suptitle('Audio Signal Analysis', fontsize=14, fontweight='bold')
    fig.tight_layout()
    plt.savefig(PLOTS_DIR / '12_audio_waveform.png', dpi=150, bbox_inches='tight')
    plt.show()

Skipping preview — librosa not available or no audio files found.


*▸ **Output above:** The waveform should show amplitude variations (speech produces distinct peaks during phonemes).  
A completely flat waveform suggests a silent or corrupted recording.  
The mel spectrogram maps energy across mel-scaled frequency bands over time — spoken words show bright bands at characteristic frequencies.*

## Step 4 — Audio Augmentation

Augmentation creates training diversity from limited recordings.  
Three standard transformations are applied:

| Augmentation | Method | Effect |
|---|---|---|
| **Pitch shift** | Shift by ±2 semitones | Simulates different vocal registers |
| **Time stretch** | Speed up/slow down (0.9×, 1.1×) | Simulates speech rate variation |
| **Gaussian noise** | Add 0.005× amplitude noise | Simulates microphone noise |

In [4]:
def augment_audio(y, sr):
    """Return list of augmented audio variants: (name, y_aug) tuples."""
    augments = []
    augments.append(('pitch_up',     librosa.effects.pitch_shift(y, sr=sr, n_steps=2)))
    augments.append(('pitch_down',   librosa.effects.pitch_shift(y, sr=sr, n_steps=-2)))
    augments.append(('stretch_slow', librosa.effects.time_stretch(y, rate=0.9)))
    augments.append(('stretch_fast', librosa.effects.time_stretch(y, rate=1.1)))
    augments.append(('noise',        y + 0.005 * np.random.randn(len(y))))
    return augments

if not LIBROSA_AVAILABLE or not audio_paths:
    print("Skipping augmentation — librosa not available or no audio found.")
else:
    sample_path = audio_paths[0]
    y_demo, sr_demo = librosa.load(str(sample_path), sr=None)
    variants = augment_audio(y_demo, sr_demo)

    fig, axes = plt.subplots(3, 2, figsize=(14, 9))
    axes = axes.flat

    librosa.display.waveshow(y_demo, sr=sr_demo, ax=axes[0], color='#2d6a4f')
    axes[0].set_title('Original', fontweight='bold')

    for ax, (name, y_aug) in zip(axes[1:], variants):
        librosa.display.waveshow(y_aug[:len(y_demo)], sr=sr_demo, ax=ax, color='#52b788')
        ax.set_title(name, fontweight='bold')

    for ax in axes: ax.label_outer()
    fig.suptitle('Audio Augmentation Preview', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / '13_audio_augmentation.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Generated {len(variants)} augmented variants per clip.")

Skipping augmentation — librosa not available or no audio found.


*▸ **Output above:** Six waveform panels — original plus 5 augmented versions.  
Pitch-shifted versions should look nearly identical to the original (subtle timing differences).  
The noise version should look slightly fuzzier (higher floor amplitude).  
Stretched versions should be visibly longer or shorter in time.*

## Step 5 — MFCC & Spectral Feature Extraction

For each audio clip we extract a **39-dimensional feature vector**:

| Feature group | Dimensions | Description |
|---|---|---|
| **MFCC** | 13 | Mel-Frequency Cepstral Coefficients — captures timbre/phoneme patterns |
| **Spectral rolloff** | 1 | Frequency below which 85% of energy is contained |
| **Zero-crossing rate** | 1 | Rate of sign changes — proxy for noisiness / consonants |
| **Spectral centroid** | 1 | Centre of mass of the spectrum (perceived brightness) |
| **RMS energy** | 1 | Root-mean-square amplitude — loudness |
| **Spectral bandwidth** | 1 | Width of the spectrum — indicates how spread the frequency content is |

All features are computed as the **mean across time** to produce a fixed-length vector regardless of clip duration.

In [5]:
def extract_audio_features(y, sr):
    """Extract a 18-dim audio feature vector from a clip."""
    mfcc      = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13).mean(axis=1)
    rolloff   = float(librosa.feature.spectral_rolloff(y=y, sr=sr).mean())
    zcr       = float(librosa.feature.zero_crossing_rate(y).mean())
    centroid  = float(librosa.feature.spectral_centroid(y=y, sr=sr).mean())
    rms       = float(librosa.feature.rms(y=y).mean())
    bandwidth = float(librosa.feature.spectral_bandwidth(y=y, sr=sr).mean())
    feat_vec  = np.concatenate([mfcc, [rolloff, zcr, centroid, rms, bandwidth]])
    return feat_vec

if not LIBROSA_AVAILABLE or not audio_paths:
    print("Creating empty audio features CSV template.")
    n_feat = 18
    cols = ['audio_path', 'member_name', 'augmentation'] + [f'feat_{i}' for i in range(n_feat)]
    pd.DataFrame(columns=cols).to_csv(OUT_DIR / 'audio_features.csv', index=False)
    print(f"Empty template saved to {OUT_DIR / 'audio_features.csv'}")
else:
    records = []
    for audio_path in audio_paths:
        try:
            y, sr = librosa.load(str(audio_path), sr=None)
        except Exception as e:
            print(f"Warning: could not load {audio_path}: {e}")
            continue
        member_name = audio_path.parent.name

        # Original
        feat = extract_audio_features(y, sr)
        row = {'audio_path': str(audio_path), 'member_name': member_name, 'augmentation': 'original'}
        row.update({f'feat_{i}': float(feat[i]) for i in range(len(feat))})
        records.append(row)

        # Augmented
        for aug_name, y_aug in augment_audio(y, sr):
            try:
                feat = extract_audio_features(y_aug, sr)
                row = {'audio_path': str(audio_path), 'member_name': member_name, 'augmentation': aug_name}
                row.update({f'feat_{i}': float(feat[i]) for i in range(len(feat))})
                records.append(row)
            except Exception:
                pass

    features_df = pd.DataFrame(records)
    features_df.to_csv(OUT_DIR / 'audio_features.csv', index=False)
    print(f"Processed {len(audio_paths)} clips + augmentations = {len(records)} rows total")
    print(f"Feature CSV shape: {features_df.shape}")
    print(f"Saved to: {OUT_DIR / 'audio_features.csv'}")
    display(features_df.head(3))

Creating empty audio features CSV template.
Empty template saved to C:\Users\user\Desktop\kayonga-elvis task-formative2\data\processed\audio_features.csv


*▸ **Output above:** Each row in `audio_features.csv` corresponds to one audio clip (original or augmented).  
Columns: `audio_path`, `member_name`, `augmentation`, then `feat_0` through `feat_17`.  
This CSV is fed into the multimodal integration notebook (`Task4_Multimodal_Integration.ipynb`).*